# TP 5 — Du texte aux vecteurs

**Cours 5 · Fondamentaux du NLP et préparation au RAG**

Durée : 1 heure — 3 blocs.

| Bloc | Durée | Contenu |
|---|---|---|
| 1 | 15 min | Normaliser, tokeniser, filtrer |
| 2 | 20 min | BoW, TF-IDF, cosinus |
| 3 | 12 min | Embeddings pré-calculés |

**Objectif** : mesurer l'effet de chaque choix de représentation, et provoquer volontairement les cas d'échec.


In [1]:
import re, unicodedata, math
from pathlib import Path

DATA = Path("data")
corpus = [l.strip() for l in (DATA / "corpus_fr.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
stopwords = set((DATA / "stopwords_fr.txt").read_text(encoding="utf-8").split())

print(len(corpus), "textes |", sum(len(t.split()) for t in corpus), "mots")
corpus[:2]


30 textes | 282 mots


## Bloc 1 — Normaliser et tokeniser (15 min)

Deux fonctions à compléter, puis une mesure : combien de formes distinctes avant et après filtrage ?

### 1.1 Normalisation

Écrire `normalize(texte)` qui :
1. passe en minuscules,
2. retire les accents (NFD puis suppression des marques combinantes),
3. remplace toute ponctuation par une espace.

*Indice* : `unicodedata.normalize("NFD", s)` puis `unicodedata.category(c) != "Mn"`.


In [2]:
def normalize(texte: str) -> str:
    texte = texte.lower()
    texte = unicodedata.normalize("NFD", texte)
    texte = "".join(c for c in texte if unicodedata.category(c) != "Mn")
    texte = re.sub(r"[^a-z0-9]+", " ", texte)
    return texte.strip()


### 1.2 Tokenisation

Écrire `tokenize(texte)` qui applique `normalize` puis découpe sur `\\w+` et supprime les tokens d'un seul caractère.


In [3]:
def tokenize(texte: str) -> list[str]:
    return [t for t in re.findall(r"\w+", normalize(texte)) if len(t) > 1]


### 1.3 Mesure

Comparer : nombre total de tokens, nombre de formes distinctes, puis le même calcul après retrait des stop words.


In [4]:
tokens = [tokenize(t) for t in corpus]
flat = [t for doc in tokens for t in doc]
lexique = set(flat)
sans_stop = [t for t in flat if t not in stopwords]
lexique_filtre = set(sans_stop)

print(f"tokens           : {len(flat)}")
print(f"formes distinctes: {len(lexique)}")
print(f"après stop words : {len(sans_stop)} tokens, {len(lexique_filtre)} formes")
print(f"réduction du lexique : {100*(1-len(lexique_filtre)/len(lexique)):.1f} %")
print(sorted(lexique - lexique_filtre)[:15])


tokens           : 278
formes distinctes: 173
après stop words : 168 tokens, 151 formes
réduction du lexique : 12.7 %
['au', 'aux', 'avec', 'ce', 'dans', 'de', 'des', 'du', 'en', 'est', 'et', 'la', 'le', 'les', 'pour']


**Questions**

1. Pourquoi `len(flat)` est-il très supérieur à `len(lexique)` ?
2. Le retrait des stop words réduit-il surtout le nombre de *tokens* ou de *formes* ?
3. Que se passerait-il si l'on retirait les stop words **avant** de tokeniser ?


## Bloc 2 — BoW, TF-IDF, cosinus (20 min)

### 2.1 Sac de mots

Construire la matrice documents × termes avec `CountVectorizer`, puis classer une requête par similarité cosinus.


In [5]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = tokenize_corpus = [" ".join(tokenize(t)) for t in corpus]

vec = CountVectorizer(stop_words=list(stopwords))
X = vec.fit_transform(docs)
print("matrice :", X.shape, "| densité :", f"{X.nnz / (X.shape[0]*X.shape[1]):.2%}")

requete = "le chat dort sur le canapé"
q = vec.transform([requete])
scores = cosine_similarity(q, X)[0]
for i in scores.argsort()[::-1][:3]:
    print(f"{scores[i]:.3f}  {corpus[i]}")
print("mots de la requête reconnus :", [w for w in tokenize(requete) if w in vec.vocabulary_])


matrice : (30, 151) | densité : 3.71%
0.535  Le chat dort sur le canapé du salon, près de la fenêtre ouverte.
0.316  Un chat de gouttière traverse la rue sans se presser.
0.289  Le chien du voisin dort toute la journée dans le jardin.
mots de la requête reconnus : ['chat', 'dort', 'canape']


In [6]:
tf = TfidfVectorizer(stop_words=list(stopwords))
Xt = tf.fit_transform(docs)
qt = tf.transform([requete])
scores_tf = cosine_similarity(qt, Xt)[0]
for i in scores_tf.argsort()[::-1][:3]:
    print(f"{scores_tf[i]:.3f}  {corpus[i]}")


0.471  Le chat dort sur le canapé du salon, près de la fenêtre ouverte.
0.300  Le chien du voisin dort toute la journée dans le jardin.
0.234  Un chat de gouttière traverse la rue sans se presser.


In [7]:
requete_inconnue = "gâteau au chocolat"
q2 = vec.transform([requete_inconnue])
print("mots reconnus :", [w for w in tokenize(requete_inconnue) if w in vec.vocabulary_])
import numpy as np
print("norme du vecteur requête :", float(np.sqrt(q2.multiply(q2).sum())))
print("scores :", cosine_similarity(q2, X)[0][:5].round(3))


mots reconnus : []
norme du vecteur requête : 0.0
scores : [0. 0. 0. 0. 0.]


**Questions**

1. La requête contient quatre mots ; combien sont réellement utilisés par le modèle ? Pourquoi ?
2. Le classement change-t-il entre comptages et TF-IDF ? De combien ?
3. Que vaut la norme du vecteur de la requête inconnue ? Que renvoie alors le cosinus, et pourquoi est-ce une convention ?


## Bloc 3 — Embeddings pré-calculés (12 min)

`data/embeddings_fr_mini.vec` contient des vecteurs de mots pré-entraînés (format texte word2vec).
Aucun entraînement : on **inspecte** un modèle existant.


In [8]:
from gensim.models import KeyedVectors

kv = KeyedVectors.load_word2vec_format(DATA / "embeddings_fr_mini.vec", binary=False)
print("vocabulaire :", len(kv.key_to_index), "| dimension :", kv.vector_size)
print("'chat' est connu :", "chat" in kv.key_to_index)


vocabulaire : 5000 | dimension : 100
'chat' est connu : True


In [9]:
for mot in ["chat", "voiture"]:
    if mot in kv.key_to_index:
        print(mot, "->", [(w, round(s, 3)) for w, s in kv.most_similar(mot, topn=5)])
    else:
        print(mot, "absent du vocabulaire")


chat -> [('félin', 0.717), ('chien', 0.618), ('chats', 0.615), ('lapin', 0.534), ('cochon', 0.532)]
voiture -> [('véhicule', 0.739), ('bagnole', 0.707), ('moto', 0.707), ('auto', 0.646), ('rouler', 0.642)]


In [10]:
def analogie(a, b, c):
    return [(w, round(s, 3)) for w, s in kv.most_similar(positive=[b, c], negative=[a], topn=3)]

for triplet in [("homme", "roi", "femme"), ("paris", "france", "italie")]:
    try:
        print(triplet, "->", analogie(*triplet))
    except KeyError as e:
        print(triplet, "-> mot absent :", e)


('homme', 'roi', 'femme') -> [('roy', 0.74), ('reine', 0.604), ('trône', 0.593)]
('paris', 'france', 'italie') -> [('espagne', 0.814), ('afrique', 0.745), ('europe', 0.735)]


In [11]:
# un échec à expliquer
for mot in ["avocat", "banque", "ordinateur"]:
    if mot in kv.key_to_index:
        print(mot, "->", [(w, round(s, 3)) for w, s in kv.most_similar(mot, topn=5)])


avocat -> [('légiste', 0.767), ('avocats', 0.761), ('avocate', 0.696), ('procureur', 0.663), ('sénateur', 0.646)]
banque -> [('monnaie', 0.621), ('caisse', 0.602), ('crédit', 0.596), ('portefeuille', 0.564), ('société', 0.563)]
ordinateur -> [('appareil', 0.822), ('portable', 0.803), ('écran', 0.714), ('ordinateurs', 0.71), ('serveur', 0.683)]


**Questions**

1. Les voisins de `chat` sont-ils tous des animaux ? Sinon, pourquoi ?
2. `avocat` a-t-il un seul sens dans cet espace ? Que faudrait-il pour distinguer les deux ?
3. Comparez le cosinus de `chat`/`chien` ici avec le cosinus TF-IDF de deux textes contenant l'un `chat`, l'autre `chien`. Que vient de changer l'embedding ?


## Conclusion

Compléter le tableau :

| Représentation | Dimensions | Sens capté | Ordre capté | Mots inconnus | Cas d'usage |
|---|---|---|---|---|---|
| BoW | | | | | |
| TF-IDF | | | | | |
| Word2Vec | | | | | |

**Question de sortie** : pour une recherche dans un catalogue qui écrit `automobile` et dont les utilisateurs tapent `voiture`, quelle représentation choisissez-vous, et quelle limite acceptez-vous ?
